# bottleneck-latent-projection — worked example 2: Mirror the bottleneck in the decoder to expand latent back to a feature map

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bottleneck-latent-projection`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The decoder mirrors the encoder bottleneck in reverse: `Linear(latent, hidden)` with ReLU, then `Linear(hidden, C*H*W)`, then `Rearrange('b (c h w) -> b c h w')` to restore the conv feature-map shape the transposed-conv stack expects. The expansion Linears bring the latent code back up to flattened-feature dimensionality before reshaping.

## Worked solution

**Step 1 - first expand Linear + ReLU.** Starting from `z` of shape `(B, latent)`, we apply `z @ W1.T + b1` to map `latent -> hidden`, then `relu`. This mirrors the encoder's final-but-one stage and reintroduces non-linear capacity on the way up.

**Step 2 - second expand Linear, no activation.** `h @ W2.T + b2` maps `hidden -> C*H*W`. As on the encoder side, the projection that produces the pre-image features carries no activation so reconstructed values can be negative.

**Step 3 - un-flatten.** The decoder's conv-transpose layers need `(B, C, H, W)`, so we invert the encoder's flatten with `rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)`. We must pass the original `c/h/w` sizes so einops knows how to split the single `(c h w)` axis - the product alone is ambiguous.

**Step 4 - check.** We feed a `(B=3, latent=4)` code through and print the output shape `(3, 8, 4, 4)`, confirming the decoder restored the encoder's feature-map geometry.

In [ ]:
def decode_bottleneck(z, W1, b1, W2, b2, C, H, W):
    h = t.relu(z @ W1.T + b1)
    flat = h @ W2.T + b2
    x = rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)
    return x

t.manual_seed(0)
B, C, H, Wd = 3, 8, 4, 4
latent, hidden = 4, 32
out_features = C * H * Wd
z = t.randn(B, latent)
W1 = t.randn(hidden, latent)
b1 = t.randn(hidden)
W2 = t.randn(out_features, hidden)
b2 = t.randn(out_features)
x = decode_bottleneck(z, W1, b1, W2, b2, C, H, Wd)
print('feature-map shape:', tuple(x.shape))
print('mean value:', round(x.mean().item(), 4))